In [1]:
from gerrychain import (GeographicPartition, Partition, Graph, MarkovChain,
                        proposals, updaters, constraints, accept, Election)

from gerrychain.proposals import recom, propose_random_flip

from gerrychain.tree import recursive_tree_part, recursive_seed_part

from gerrychain.metrics import efficiency_gap, mean_median, polsby_popper, partisan_bias

from gerrychain.updaters import cut_edges

from gerrychain.tree import bipartition_tree, find_balanced_edge_cuts_memoization


import geopandas as gpd
import matplotlib.pyplot as plt

import networkx as nx




In [2]:
graph = Graph.from_json("./AL_Processed_Precincts.json")
df = gpd.read_file("./AL_Processed_Precincts.shp")


In [3]:
def count_spanning(graph):
    laplacian = nx.laplacian_matrix(graph)
    L = np.delete(np.delete(laplacian.todense(), 0, 0), 1, 1)
    return np.linalg.slogdet(L)[1]

def county_splits(partition, df=df):
    df["current"] = df["PRECINCTID"].map(partition.assignment)

    counties = sum(df.groupby("COUNTYFP")['current'].nunique()>1)
    return counties

election_names = [
    "PRE"
]

num_elections = len(election_names)

election_columns = [
['G24PRERTRU','G24PREDHAR']
]

my_updaters = {
    "population": updaters.Tally("population", alias="population"),
    "cut_edges": cut_edges,
    "PP":polsby_popper,
    "county_splits": county_splits
}

elections = [
    Election(
        election_names[i],
        {"Democratic": election_columns[i][1], "Republican": election_columns[i][0]},
    )
    for i in range(num_elections)
]

election_updaters = {election.name: election for election in elections}

for node in graph.nodes():
    graph.nodes()[node]["non_NH_Black"] = graph.nodes()[node]["population"] - graph.nodes()[node]["NH_Black"]
    graph.nodes()[node]["non_Hispanic"] = graph.nodes()[node]["population"] - graph.nodes()[node]["Hispanic"]

my_updaters.update({"NH_Black":Election("NH_Black",{"NH_Black": "NH_Black", "non_NH_Black": "non_NH_Black"})})
my_updaters.update({"Hispanic":Election("Hispanic",{"Hispanic": "Hispanic", "non_Hispanic": "non_Hispanic"})})

# save percentages

my_updaters.update(election_updaters)

In [4]:
import json

with open("./AL_assignment_30.json", "r") as file:
    restart_dict = json.load(file)

In [5]:
restart_partition = GeographicPartition(graph,restart_dict, my_updaters)

In [7]:
def pp_constraint(partition): 

    return sum([1/x for x in polsby_popper(partition).values()])/7 > 4


def county_constraint(partition):

    return partition['county_splits'] < 11


def competitiveness_constraint(partition):

    return sum([abs(x-.5)<.05 for x in partition['PRE'].percents("Democratic")]) > 0





In [8]:
from functools import partial 

ideal_population = df['population'].sum()/7

county_proposal = partial(
    recom,
    pop_col="population",
    pop_target=ideal_population,
    epsilon=0.03,
    node_repeats=2,
    region_surcharge = {"COUNTYFP":1}
)

second_recom_chain = MarkovChain(
    proposal=county_proposal,
    constraints=[pp_constraint],
    accept=accept.always_accept,
    initial_state=restart_partition,
    total_steps=70 #number of remaining steps to your target
)

In [ ]:

cs = []
mms = []
egs = []
pbs =[]
dvp = []
pps = []
bvp = []
mbvp = []
wins = []
cds = []


temp = 0

for part in second_recom_chain:

    temp += 1

    if temp %10 == 0:
        
        print("Step Number", temp)
        ad = dict(part.assignment)

        with open(f"AL_assignment_{temp}.json", "w") as file:
            json.dump(ad, file)


        plt.figure(figsize=(4,10))
        nx.draw(graph, pos = {x:(graph.nodes()[x]['C_X'],graph.nodes()[x]['C_Y']) for x in graph.nodes()},node_color=[ad[x] for x in graph.nodes()],
            cmap='tab20b',node_size=15)
        plt.savefig(f'./network_plot_{temp}.png')
        plt.close()
    
        df['current'] = df["PRECINCTID"].map(ad)
        df.plot(column='current',cmap='tab20b')
        plt.axis('off')
        plt.savefig(f'./df_plot_{temp}.png')
        plt.close()


        ndf = pd.DataFrame({"CountySplits":cs, "MM":mms, 'EG':egs,'PB':pbs,'DWins':wins,'PP':pps,'Comp45-55':cds})

        ndf.to_csv(f"./chain_outputs_{temp}.csv")
        
        with open(f"./DemPercs_{temp}.csv", "w") as tf1:
            writer = csv.writer(tf1, lineterminator="\n")
            writer.writerows(dvp)
        
        
        with open(f"./BlackPercs_{temp}.csv", "w") as tf1:
            writer = csv.writer(tf1, lineterminator="\n")
            writer.writerows(bvp)

        cs = []
        mms = []
        egs = []
        pbs =[]
        dvp = []
        pps = []
        bvp = []
        mbvp = []
        wins = []
        cds = []
    
    

    cs.append(part['county_splits'])
    mms.append(mean_median(part['PRE']))
    egs.append(efficiency_gap(part['PRE']))
    pbs.append(partisan_bias(part['PRE']))
    dvp.append(sorted(part['PRE'].percents("Democratic")))
    pps.append(sum([1/x for x in polsby_popper(part).values()])/7)
    bvp.append(sorted(part['NH_Black'].percents("NH_Black")))
    mbvp.append(max(bvp[-1]))
    wins.append(part['PRE'].wins("Democratic"))
    cds.append(sum([abs(x-.5)<.05 for x in part['PRE'].percents("Democratic")]))





